# Assignment 4 - Train a ternary text-classification model

**Due Date: 2025.05.27 23:59**

For this assignment you can work by yourself or in pair.

### Task

The goal of this assignment is to train a text classification model with Spacy, based on the provided `financial-sentiment.csv` dataset. Such dataset is composed of two columns: one with the text, and one with the sentiment of the text, which can be positive, neutral, or negative.

Your task is to use the provided dataset to train a text classifier in Spacy, using either the Spacy 2 (write the code in a notebook), or Spacy 3 convention (train the Spacy model from the terminal). We have seen in class (with examples) how to do both, so you can refer to the notebooks and/or the video of the lectures if you don't remember all the details.

Remember that to use the Spacy 3 convention, you need to convert the data into DocBin (look at examples notebook to see how to do that).

Once you created and trained the model, you should apply it on the following sentences:
- the earnings were way above expectations
- tesla plans to double its production within a year
- the latest FED policy will push have a great impact on all stocks
- the latest FED policy will push have a great impact on all stocks but this one
- unclear what will happen in the next quarter
- earning flop? look at the details before drawing any conclusion
- it is impossible to time the market, but nonetheless this stock looks like a sell

To print the result, you can use the following snippet:

In [36]:
def print_prediction_results(docs, tc):
    for doc in docs:
        result = tc.predict([doc])
        print(f"{doc.doc}")
        for l,r in zip(tc.labels, result[0]):
            print("{}   \t{:.2f}%".format(l, r*100))
        print("===========================")


where `tc` is the text classifier that you can extract from the nlp model as follows:

```
tc = nlp.get_pipe('textcat')
```

and `docs` is the list of Spacy documents that you can create from the given sentences as follows:

```
sentences = [
    "one sentence",
    "another sentence",
    ...
]

docs = [ nlp.make_doc(sentence) for sentence in sentences ]
```


In addition, for each sentence listed above, print "positive", "negative", or "neutral" based on the result of the prediction. For example, if the sentence `"the earnings were way above expectations"` is 0.4 positive, 0.3 negative, and 0.3 neutral you should print:

```
the earnings were way above expectations
sentiment: positive
```

### Deliverable

The deliverable of the assignment is a zip file (named `lastname.zip` if you work alone, or named `lastname1_lastname2.zip` if you work in pairs), containing the following:
- the notebook you used to convert the data in Spacy format
- the Spacy configuration file (`config.cfg`)
- a description of the steps that you followed to create the configuration and train the Spacy model
- the final performance of your trained model (EPOCH, ITERATION, LOSS TEXTCAT, CATS_SCORE, CATS_MICRO_P, CATS_MICRO_R, SCORE)
- the notebook you used to apply the model on the sentences listed above

**Note:** the `financial-sentiment.csv` dataset is without the header row and the encoding is not UTF8 but windows-1252. To import the dataset as a Pandas dataframe, you can use the following snippet:

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
import spacy
from spacy.tokens import DocBin

# Load the dataset
df = pd.read_csv("financial-sentiment.csv", encoding="windows-1252", header=None)
df = df.rename(columns={0: "sentiment", 1: "news"})

# Display sentiment distribution
print("Sentiment distribution:")
print(df['sentiment'].value_counts())
print("\\n")

# Separate features and target
X = df['news']
y = df['sentiment']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)
print("\\n")

# Prepare data for spaCy
# We will create a list of tuples (text, {"cats": {"label": 1.0 or 0.0}})
# For multi-class, it will be like: (text, {"cats": {"positive": 1.0, "negative": 0.0, "neutral": 0.0}})

def create_spacy_data(texts, labels, sentiment_map):
    data = []
    for text, label in zip(texts, labels):
        cats = {sentiment: 0.0 for sentiment in sentiment_map.keys()}
        if label in sentiment_map: # Ensure the label is one we are considering
            cats[label] = 1.0
        data.append((text, {"cats": cats}))
    return data

sentiment_map = {"positive": "positive", "neutral": "neutral", "negative": "negative"}

train_data = create_spacy_data(X_train, y_train, sentiment_map)
test_data = create_spacy_data(X_test, y_test, sentiment_map)

# Let's check the first few entries
print("Sample of training data:")
for item in train_data[:2]:
    print(item)
print("\\n")

print("Sample of testing data:")
for item in test_data[:2]:
    print(item)

# Convert to DocBin format (optional for spaCy 2 style training in notebook, but good practice and required for spaCy 3 CLI)
# We will create separate DocBin objects for train and dev (test) sets

nlp = spacy.blank("en") # Load a blank English model

def convert_to_docbin(data, filename):
    db = DocBin()
    for text, annotations in data:
        doc = nlp.make_doc(text)
        doc.cats = annotations["cats"]
        db.add(doc)
    db.to_disk(filename)
    print(f"Data converted and saved to {filename}")

convert_to_docbin(train_data, "./train.spacy")
convert_to_docbin(test_data, "./dev.spacy") # spaCy often refers to the test/validation set as 'dev'


Sentiment distribution:
sentiment
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64
\n
Training set shape: (3876,)
Testing set shape: (970,)
\n
Sample of training data:
('The major breweries increased their domestic beer sales by 4.5 per cent last year , to 256.88 million litres from 245.92 million litres in 2004 .', {'cats': {'positive': 1.0, 'neutral': 0.0, 'negative': 0.0}})
('CapMan , an asset manager , has EUR 3bn worth of assets under management in the Nordic region .', {'cats': {'positive': 0.0, 'neutral': 1.0, 'negative': 0.0}})
\n
Sample of testing data:
('Following the payment made in April , the company has a total of EUR 23.0 million in loans from financial institutions .', {'cats': {'positive': 0.0, 'neutral': 1.0, 'negative': 0.0}})
('The share subscription period for C options will commence on 1 September 2008 and expire on 31 March 2011 .', {'cats': {'positive': 0.0, 'neutral': 1.0, 'negative': 0.0}})
Data converted and saved to ./train.spacy


In [ ]:
import spacy

nlp_model_path = "/Users/zitian/Text-Analysis-and-Spatial-Data-for-Economists-SP-2025/assignment4/output/model-best" 
print(f"Loading model from: {nlp_model_path}")
nlp = None # Initialize nlp to None
try:
    nlp = spacy.load(nlp_model_path)
    print("Model loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure the model path is correct and the training was successful.")
    print("Common issues: incorrect path, or model training did not complete successfully.")

# Sentences to predict as per assignment instructions
sentences = [
    "the earnings were way above expectations",
    "tesla plans to double its production within a year",
    "the latest FED policy will push have a great impact on all stocks", # Note original typo
    "the latest FED policy will push have a great impact on all stocks but this one", # Note original typo
    "unclear what will happen in the next quarter",
    "earning flop? look at the details before drawing any conclusion",
    "it is impossible to time the market, but nonetheless this stock looks like a sell"
]


print("\n--- Applying model to sentences and printing results ---")
if nlp is not None:
    textcat = None
    if "textcat" in nlp.pipe_names:
        textcat = nlp.get_pipe("textcat")
        # The labels from textcat.labels could be used to ensure a specific order if required
        # print(f"Textcat labels from model: {textcat.labels}")
    else:
        print("ERROR: 'textcat' component not found in the loaded model pipeline.")

    if textcat: # Proceed only if textcat component is found
        print("\nProcessing and printing predictions as per assignment requirements:")
        for sentence_text in sentences:
            # Process the current sentence text with the loaded nlp pipeline
            # This will populate doc.cats with the prediction scores
            doc = nlp(sentence_text) 
            
            print(f"\n{doc.text}") # Print the sentence (as required before probabilities)
            
            for label in textcat.labels: # Iterate in the order of model's known labels
                score = doc.cats.get(label, 0.0) # Get score, default to 0.0 if somehow missing
                print(f"{label: <10}\t{score*100:.2f}%") # Match assignment format
            
            # Determine and print the final sentiment (positive, negative, or neutral)
            if doc.cats:
                # Find the label with the highest score in doc.cats
                predicted_sentiment = max(doc.cats, key=doc.cats.get)
                # Print the simplified sentiment as required by the assignment
                print(f"sentiment: {predicted_sentiment}") # Matching "sentiment: positive" format
            else:
                print("sentiment: Could not be determined (no categories returned).")
            print("-------------------------------------------") # Separator as per your example
            
else:
    print("NLP model ('nlp') was not loaded or failed to load. Cannot proceed with predictions.")


Loading model from: /Users/zitian/Text-Analysis-and-Spatial-Data-for-Economists-SP-2025/assignment4/output/model-best
Model loaded successfully.

--- Applying model to sentences and printing results ---

Processing and printing predictions as per assignment requirements:

the earnings were way above expectations
positive  	100.00%
neutral   	0.00%
negative  	0.00%
sentiment: positive
-------------------------------------------

tesla plans to double its production within a year
positive  	99.98%
neutral   	0.02%
negative  	0.00%
sentiment: positive
-------------------------------------------

the latest FED policy will push have a great impact on all stocks
positive  	0.95%
neutral   	99.05%
negative  	0.00%
sentiment: neutral
-------------------------------------------

the latest FED policy will push have a great impact on all stocks but this one
positive  	0.47%
neutral   	99.53%
negative  	0.00%
sentiment: neutral
-------------------------------------------

unclear what will happe

In [13]:
import pandas as pd # Make sure pandas is imported

# This code should run AFTER the previous cell where 'nlp' is loaded
# and 'sentences' are defined.

print("\n--- Creating Summary Table of Predictions ---")

if 'nlp' in locals() and nlp is not None:
    if "textcat" not in nlp.pipe_names:
        print("ERROR: 'textcat' component not found. Cannot generate summary table.")
    else:
        textcat_component = nlp.get_pipe("textcat")
        textcat_labels = textcat_component.labels # Get labels e.g., ['positive', 'neutral', 'negative']
        
        predictions_data = []

        for sentence_text in sentences: # 'sentences' list should be from the previous cell
            doc = nlp(sentence_text)
            
            # Prepare data for the current sentence
            row_data = {'Sentence': doc.text}
            
            # Add probabilities for each label
            for label in textcat_labels:
                row_data[f'Prob_{label}'] = doc.cats.get(label, 0.0) * 100 # Store as percentage
            
            # Determine and add the final predicted sentiment
            if doc.cats:
                predicted_sentiment = max(doc.cats, key=doc.cats.get)
                row_data['Predicted_Sentiment'] = predicted_sentiment
            else:
                row_data['Predicted_Sentiment'] = "N/A" # Should not happen if model is working
                
            predictions_data.append(row_data)

        # Create a Pandas DataFrame
        if predictions_data:
            df_predictions = pd.DataFrame(predictions_data)
            
            # Define the order of columns for better readability
            column_order = ['Sentence']
            for label in textcat_labels: # Add probability columns in the order of model labels
                column_order.append(f'Prob_{label}')
            column_order.append('Predicted_Sentiment')
            
            # Reorder DataFrame columns
            df_predictions = df_predictions[column_order]

            # Display the DataFrame
            # In a Jupyter Notebook, simply calling the DataFrame will display it nicely.
            print("Summary of Predictions:")
            
            from IPython.display import display
            display(df_predictions)
        else:
            print("No prediction data was generated to create a table.")
else:
    print("NLP model ('nlp') was not loaded. Cannot generate summary table.")



--- Creating Summary Table of Predictions ---
Summary of Predictions:


,Sentence,Prob_positive,Prob_neutral,Prob_negative,Predicted_Sentiment
0,the earnings were way above expectations,1.000000e+02,0.000002,1.145529e-09,positive
1,tesla plans to double its production within a ...,9.997806e+01,0.021929,1.043231e-05,positive
2,the latest FED policy will push have a great i...,9.453593e-01,99.054623,2.753681e-05,neutral
3,the latest FED policy will push have a great i...,4.696617e-01,99.530327,1.506924e-05,neutral
4,unclear what will happen in the next quarter,1.756271e-07,19.250177,8.074982e+01,negative
5,earning flop? look at the details before drawi...,4.693295e-12,99.999940,6.022022e-05,neutral
6,"it is impossible to time the market, but nonet...",1.939890e-07,100.000000,2.206116e-12,neutral
